# ARCHS4 reconstruction error — saturation models, random subsamples (compute)

**Environment:** `clamp-analyses`

Evaluates how well CLAMPfull_hall reconstructs gene expression for saturation models (fixed k, varying coverage) trained on **random subsamples**.

Dynamically discovers all directories under `07_saturation/`. Any directory missing `CLAMPfull_hall.rds` is skipped — re-run the notebook once those jobs finish to pick them up.

For each available directory:
1. Load CLAMPfull_hall model from `07_saturation`
2. Look up the matching FBM from `06_bp_coverage_rshall` (matched by coverage % and seed index)
3. Row-normalise expression data (z-score per gene)
4. Reconstruct with `Z %*% B`
5. Compute per-sample Spearman correlation

Inputs:
- `output/01_model_building/04_archs4/07_saturation/` (models)
- `output/01_model_building/04_archs4/06_bp_coverage_rshall/` (FBMs)

Outputs: `output/01_model_building/06_reconstruction_error/`

In [ ]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(here)

source(here("config.R"))

## Helper functions

In [ ]:
get_chunked_cors <- function(X_fbm, Z, B,
                              chunk_size = 500, cor.method = "spearman") {
  n_samples <- ncol(X_fbm)
  cors      <- numeric(n_samples)
  for (start in seq(1, n_samples, by = chunk_size)) {
    end        <- min(start + chunk_size - 1L, n_samples)
    idx        <- start:end
    chunk      <- X_fbm[, idx, drop = FALSE]
    xhat_chunk <- Z %*% B[, idx, drop = FALSE]
    for (j in seq_along(idx)) {
      cors[idx[j]] <- cor(chunk[, j], xhat_chunk[, j], method = cor.method)
    }
    rm(chunk, xhat_chunk)
  }
  cors
}

## Parameters

In [ ]:
base_rshall_dir <- file.path(config$ARCHS4$DATASET_FOLDER, "06_bp_coverage_rshall")
base_sat_dir    <- file.path(config$ARCHS4$DATASET_FOLDER, "07_saturation")
output_dir      <- here("output", "01_model_building", "06_reconstruction_error")
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

message("Output dir:     ", output_dir)
message("Saturation dir: ", base_sat_dir)
message("Random FBM dir: ", base_rshall_dir)

## Main loop: compute reconstruction correlation

In [ ]:
results_path <- file.path(output_dir, "reconstruction_error_random_saturation_results.rds")

if (file.exists(results_path)) {
  message("Results already exist, loading from: ", results_path)
  results_list <- readRDS(results_path)
  message("Loaded ", length(results_list), " entries")
} else {
  # --- Build FBM lookup: key = "cov{pct}_run{idx}" -> fbm backing file path ---
  fbm_lookup <- list()
  for (pct_dir in list.dirs(base_rshall_dir, recursive = FALSE, full.names = TRUE)) {
    for (seed_dir in list.dirs(pct_dir, recursive = FALSE, full.names = TRUE)) {
      bk_file   <- paste0(file.path(seed_dir, "fbm_subsampled"), ".bk")
      info_file <- file.path(seed_dir, "subsample_info.rds")
      if (!file.exists(bk_file) || !file.exists(info_file)) next
      sub_info <- readRDS(info_file)
      key <- paste0("cov", round(sub_info$coverage * 100), "_run", sub_info$run)
      fbm_lookup[[key]] <- file.path(seed_dir, "fbm_subsampled")
    }
  }
  message("FBM lookup built: ", length(fbm_lookup), " entries")

  results_list <- list()

  # --- Dynamically discover all saturation dirs ---
  sat_dirs <- list.dirs(base_sat_dir, recursive = FALSE, full.names = TRUE)

  for (sat_dir in sat_dirs) {
    clamp_path <- file.path(sat_dir, "CLAMPfull_hall.rds")

    message(strrep("=", 60))
    message("Dir: ", basename(sat_dir))

    if (!file.exists(clamp_path)) {
      message("  Skipping (model not ready): ", basename(sat_dir))
      next
    }

    tryCatch({
      run_info     <- readRDS(file.path(sat_dir, "run_info.rds"))
      coverage_pct <- round(run_info$coverage * 100)
      run_idx      <- run_info$seed_idx
      k_target     <- run_info$CLAMP_K
      current_seed <- run_info$seed

      key      <- paste0("cov", coverage_pct, "_run", run_idx)
      fbm_path <- fbm_lookup[[key]]
      if (is.null(fbm_path)) {
        message("  Skipping (FBM not found for key=", key, ")")
        next
      }

      clamp_full <- readRDS(clamp_path)
      Z_full     <- as.matrix(clamp_full$Z)
      B_full     <- as.matrix(clamp_full$B)
      n_genes    <- nrow(Z_full)
      n_samples  <- ncol(B_full)
      message("  Coverage: ", coverage_pct, "% | k_target: ", k_target,
              " | Run: ", run_idx,
              " | ", n_genes, " genes x ", n_samples, " samples")

      X_fbm <- FBM(
        nrow        = n_genes,
        ncol        = n_samples,
        type        = "double",
        backingfile = fbm_path,
        create_bk   = FALSE
      )

      message("  Reconstructing (chunked)...")
      cors_full <- get_chunked_cors(X_fbm, Z_full, B_full, chunk_size = 500L)
      message("  Median cor: ", round(median(cors_full, na.rm = TRUE), 4))

      results_list[[length(results_list) + 1]] <- list(
        coverage_pct = coverage_pct,
        seed         = current_seed,
        run          = run_idx,
        model        = "CLAMPfull_hall",
        sampling     = "random_saturation",
        k_target     = k_target,
        n_lvs        = ncol(Z_full),
        n_samples    = n_samples,
        median_cor   = median(cors_full, na.rm = TRUE),
        mean_cor     = mean(cors_full, na.rm = TRUE),
        cor_vector   = list(cors_full)
      )

      rm(clamp_full, X_fbm, Z_full, B_full, cors_full)
      gc()
    }, error = function(e) {
      message("  ERROR (skipping): ", conditionMessage(e))
    })
  }

  message("\nAll available runs complete.")
}

## Save results

In [ ]:
results_df <- dplyr::bind_rows(
  lapply(results_list, function(x) {
    data.frame(
      coverage_pct = x$coverage_pct,
      seed         = x$seed,
      run          = x$run,
      model        = x$model,
      sampling     = x$sampling,
      k_target     = x$k_target,
      n_lvs        = x$n_lvs,
      n_samples    = x$n_samples,
      median_cor   = x$median_cor,
      mean_cor     = x$mean_cor,
      stringsAsFactors = FALSE
    )
  })
)

saveRDS(results_list, file.path(output_dir, "reconstruction_error_random_saturation_results.rds"))
data.table::fwrite(results_df, file.path(output_dir, "reconstruction_error_random_saturation_summary.csv"))

message("Saved to ", output_dir)
print(results_df)